# XAI-IDS: Explainable AI Intrusion Detection System
## Exploration and Demo Notebook

This notebook provides an interactive walkthrough of the XAI-IDS pipeline for the CIC-IDS-2017 dataset.

**Authors:** Mohammad Thabet Hassan, Fahad Sadek, Ahmed Sami  
**Supervisor:** Dr. Mehak Khurana  
**Dataset:** CIC-IDS-2017 (Canadian Institute for Cybersecurity)


In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

# Add project root to path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

plt.rcParams.update({"figure.dpi": 100, "font.size": 10})
print(f"Project root: {PROJECT_ROOT}")


## 1. Dataset Overview

The CIC-IDS-2017 dataset contains labeled network traffic with 78 flow-level features and 15 traffic classes (1 benign + 14 attack types). Below we load and explore the dataset.


In [ ]:
# Load dataset
df = pd.read_csv("data/raw/sample_cicids2017.csv")
print(f"Dataset shape: {df.shape}")
print(f"Columns: {len(df.columns)}")
df.head()


In [ ]:
# Class distribution
print("Class distribution:")
print(df["Label"].value_counts())
print(f"
Total samples: {len(df)}")
print(f"Number of classes: {df['Label'].nunique()}")


## 2. Data Visualization


In [ ]:
# Class distribution bar chart
fig, ax = plt.subplots(figsize=(12, 6))
class_counts = df["Label"].value_counts()
colors = plt.cm.Set3(np.linspace(0, 1, len(class_counts)))
class_counts.plot(kind="barh", ax=ax, color=colors, edgecolor="white")
ax.set_title("CIC-IDS-2017 Traffic Class Distribution", fontsize=14, fontweight="bold")
ax.set_xlabel("Number of Samples")
plt.tight_layout()
plt.show()


In [ ]:
# Feature statistics
numeric_df = df.select_dtypes(include=[np.number])
print(f"Numeric features: {numeric_df.shape[1]}")
numeric_df.describe().T.head(20)


In [ ]:
# Correlation heatmap of top features (by variance)
top_features = numeric_df.var().nlargest(20).index.tolist()
corr = numeric_df[top_features].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, cmap="RdBu_r", center=0, annot=False, square=True, ax=ax)
ax.set_title("Feature Correlation Heatmap (Top 20 by Variance)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## 3. Preprocessing Pipeline

The preprocessing pipeline handles:
- Removal of infinite and NaN values
- Duplicate removal
- Label encoding (string labels -> integers)
- Feature standardization (StandardScaler)
- Stratified train/validation/test split (70/10/20)


In [ ]:
from src.data.preprocessing import clean_data, encode_labels, split_data, scale_features

# Clean
df_clean = clean_data(df.copy())
print(f"After cleaning: {df_clean.shape}")

# Encode labels
df_enc, le, label_mapping = encode_labels(df_clean.copy(), save_path=None)
print(f"
Label mapping:")
for name, code in sorted(label_mapping.items(), key=lambda x: x[1]):
    print(f"  {code}: {name}")

# Split
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df_enc)
print(f"
Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")

# Scale
X_train_s, X_val_s, X_test_s, scaler = scale_features(X_train, X_val, X_test, save_path=None)


## 4. Model Training & Evaluation

We train three classifiers:
1. **Logistic Regression** - Linear baseline
2. **Random Forest** - Ensemble of decision trees
3. **XGBoost** - Gradient boosted trees


In [ ]:
# Load pre-computed results
metrics_df = pd.read_csv("outputs/results_metrics.csv")
print("Model Comparison:")
print("=" * 65)
metrics_df


## 5. Results Visualization

### Model Comparison


In [ ]:
display(Image(filename="outputs/figures/model_comparison.png"))


### Confusion Matrices


In [ ]:
for model in ["logistic_regression", "random_forest", "xgboost"]:
    print(f"
{'='*50}")
    print(f"  {model.replace('_', ' ').title()}")
    print(f"{'='*50}")
    display(Image(filename=f"outputs/figures/confusion_matrix_{model}_normalized.png"))


## 6. Explainability (SHAP & LIME)

### SHAP - Global Feature Importance
SHAP (SHapley Additive exPlanations) shows which features most influence the model's predictions globally.


In [ ]:
for model in ["random_forest", "xgboost"]:
    print(f"
SHAP Feature Importance - {model.replace('_', ' ').title()}")
    display(Image(filename=f"outputs/figures/shap_feature_importance_{model}.png"))


### LIME - Local Explanations
LIME explains individual predictions by fitting interpretable models around specific data points.


In [ ]:
for model in ["random_forest", "xgboost"]:
    for kind in ["correct", "misclassified"]:
        path = f"outputs/figures/lime_{kind}_{model}.png"
        if os.path.exists(path):
            print(f"
LIME {kind.title()} - {model.replace('_', ' ').title()}")
            display(Image(filename=path))


## 7. Conclusion

This notebook demonstrated the complete XAI-IDS pipeline:

1. **Data Loading & Exploration**: The CIC-IDS-2017 dataset contains 78 network flow features across 15 traffic classes
2. **Preprocessing**: Cleaning, encoding, scaling, and stratified splitting
3. **Model Training**: Three classifiers trained and compared
4. **Evaluation**: Random Forest and XGBoost outperform Logistic Regression
5. **Explainability**: SHAP reveals global feature importance; LIME explains individual predictions

### Key Findings
- Tree-based models (Random Forest, XGBoost) significantly outperform Logistic Regression
- Network flow features like packet lengths, flow duration, and IAT are most discriminative
- SHAP and LIME provide complementary views: global importance vs. local explanations

### Future Work
- Evaluate on the full CIC-IDS-2017 dataset (2.8M+ rows)
- Add deep learning models (LSTM, Transformer)
- Implement real-time inference pipeline
- Explore additional XAI techniques (Counterfactual explanations, Anchors)
